In [7]:
%load_ext autoreload
%autoreload 2
%reload_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [8]:
from datasets import load_dataset
import pandas as pd


## Loading the MGSD dataset.

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()


## Loading the MentalManip dataset

dataset_2 = load_dataset("audreyeleven/MentalManip", "mentalmanip_maj")
data_2 = dataset_2["train"]
df_2 = data_2.to_pandas()

Some datasets params were ignored: ['license']. Make sure to use only valid params for the dataset builder and to have a up-to-date version of the `datasets` library.


In [9]:
from data_loader import load_mgsd_dataset, load_mentalmanip_dataset

sample_sizes_mgsd = {
    'stereotype': 250,
    'unrelated': 250,
}

sample_size_examples_mgsd = {
    'stereotype': 5,
    'unrelated': 5
}

sample_sizes_manip = {1: 250, 0: 250}
sample_sizes_examples_manip = {1: 5, 0: 5}
max_len_examples = 1000

sample_mgsd, sample_examples_mgsd = load_mgsd_dataset(
    df, 
    sample_sizes_mgsd, 
    sample_size_examples_mgsd,
    random_state=42,
    random_state_examples=0,
    )

sample_mentalmanip, sample_examples_mentalmanip = load_mentalmanip_dataset(
    df_2, 
    sample_sizes_manip, 
    sample_sizes_examples_manip, 
    max_len_examples,
    random_state=42,
    random_state_examples=0,
    )


print("MGSD Test set balance:\n", sample_mgsd["label"].value_counts())
print("MGSD Few-shot examples balance:\n", sample_examples_mgsd["label"].value_counts())

print("MentalManip Test set balance:\n", sample_mentalmanip["manipulative"].value_counts())
print("MentalManip Few-shot examples balance:\n", sample_examples_mentalmanip["manipulative"].value_counts())

MGSD Test set balance:
 label
unrelated     250
stereotype    250
Name: count, dtype: int64
MGSD Few-shot examples balance:
 label
stereotype    5
unrelated     5
Name: count, dtype: int64
MentalManip Test set balance:
 manipulative
1    250
0    250
Name: count, dtype: int64
MentalManip Few-shot examples balance:
 manipulative
1    5
0    5
Name: count, dtype: int64


In [10]:
from dotenv import load_dotenv
import openai
import os
import json

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4.1-mini"
model_filename = "openai_4.1_mini"

## Chain of Thoughts

In [11]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from cases.mmlu_case import mmlu_case

case_name_set = ["stereotype", "manipulation"]  # "stereotype", "manipulation", "mmlu"
strategy = "optimized"          # "optimized", "zero_plus"
max_tokens = 700

try:
    for case_name in case_name_set:
        print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) ===")

        if case_name.lower() == "manipulation":
            case = manipulation_case
            task_definition = manipulation_definition_short
            data = sample_mentalmanip
        elif case_name.lower() == "stereotype":
            case = stereotypes_case
            task_definition = stereotype_definition_short_binary
            data = sample_mgsd
        else:
            raise ValueError(f"Unknown case name: {case_name}")

        cot_classifier = ChainOfThoughts(
            case=case,
            client=client,
            model=model,
            max_tokens=max_tokens,
            task_definition=task_definition,
        )

        rows = []
        detailed_reasoning = []

        for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
            text = row[case.input_col]
            true_label = row[case.label_col]
            if isinstance(true_label, str):
                true_label = true_label.strip()

            try:
                predicted_label, metrics = cot_classifier.classify_with_strategy(
                    text, strategy=strategy
                )

                mapped_label = case.label_map.get(
                    predicted_label.strip(), list(case.label_map.values())[-1]
                )

                results = {
                    "sample_id": idx,
                    "text": text,
                    "true_label": true_label,
                    "pred_label": mapped_label,
                    "raw_pred_label": predicted_label,
                    "max_tokens": cot_classifier.max_tokens,
                    "tokens_used": metrics.get("tokens_used"),
                    "prompt_tokens": metrics.get("prompt_tokens"),
                    "completion_tokens": metrics.get("completion_tokens"),
                    "latency": metrics.get("latency"),
                    "strategy": strategy,
                }
                rows.append(results)

                raw_resp = metrics.get("raw_response", "")
                steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                reasoning_detail = {
                    "sample_id": idx,
                    "raw_response": raw_resp,
                    "parsed_steps": steps,
                    "analysis": analysis,       
                    "final_line": final_line,
                    "final_label": predicted_label,
                    "mapped_label": mapped_label,
                }
                detailed_reasoning.append(reasoning_detail)

            except Exception as e:
                print(f"Error processing sample {idx}: {e}")
                continue

        if not rows:
            print(f"No successful classifications for {case_name}")
            continue

        output_file = f"results/{model_filename}/cot/classic/results_{case_name.lower()}_{strategy}_cot.csv"
        reasoning_file = f"results/{model_filename}/cot/classic/reasoning_{case_name.lower()}_{strategy}_cot.json"

        os.makedirs(os.path.dirname(output_file), exist_ok=True)

        df_out = pd.DataFrame(rows)
        df_out.to_csv(output_file, index=False)
        print(f"=== Saved {len(df_out)} rows to {output_file} ===")

        with open(reasoning_file, 'w', encoding='utf-8') as f:
            json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
        print(f"=== Saved detailed reasoning to {reasoning_file} ===")

        try:
            if case_name.lower() == "manipulation":
                y_true = df_out["true_label"].astype(int)
                y_pred = df_out["pred_label"].astype(int)
            elif case_name.lower() == "stereotype":
                y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()


            print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
            print(classification_report(y_true, y_pred, zero_division=0))
            print(f"\n=== Confusion Matrix for {case_name} ===")
            labels = sorted(set(y_true) | set(y_pred))
            print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))

            accuracy = (y_true == y_pred).mean()
            print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")

            print(f"\n=== Label Distribution ===")
            print("True labels:")
            print(pd.Series(y_true).value_counts())
            print("Predicted labels:")
            print(pd.Series(y_pred).value_counts())

        except Exception as e:
            print(f"Error in evaluation for {case_name}: {e}")

        metrics = cot_classifier.get_metrics()
        print(f"\n=== CoT Performance Metrics ===")
        print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
        print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
        print(f"- Total calls: {metrics['total_calls']}")


        print(f"\n=== Sample Reasoning (raw/parsed) ===")
        for i, reasoning in enumerate(detailed_reasoning[:3]):
            print(f"\nSample {reasoning['sample_id']}:")
            print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
            print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
            if reasoning['parsed_steps']:
                print("Parsed Steps:")
                for step in reasoning['parsed_steps']:
                    print(f"  Step {step['step']}: {step['content'][:150]}...")
            else:
                print("No explicit 'Step N' blocks found.")
            if reasoning['final_line']:
                print(f"Final Line: {reasoning['final_line']}")
            print("Raw Response (truncated):")
            print((reasoning['raw_response'] or "")[:300] + "...")
            print("-" * 50)

except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")



=== Running Chain of Thought for stereotype (optimized) ===


Processing stereotype: 100%|██████████| 500/500 [26:05<00:00,  3.13s/it]  


=== Saved 500 rows to results/openai_4.1_mini/cot/classic/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/classic/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.73      0.74      0.74       250
   unrelated       0.74      0.72      0.73       250

    accuracy                           0.73       500
   macro avg       0.73      0.73      0.73       500
weighted avg       0.73      0.73      0.73       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         186         64
unrelated           70        180

=== Accuracy for stereotype: 73.20% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    256
unrelated     244
Name: count, dtype: int64

=== CoT Perfor

Processing manipulation: 100%|██████████| 500/500 [33:43<00:00,  4.05s/it]  

=== Saved 500 rows to results/openai_4.1_mini/cot/classic/results_manipulation_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/classic/reasoning_manipulation_optimized_cot.json ===

=== Classification Report for manipulation (optimized) ===
              precision    recall  f1-score   support

           0       0.78      0.20      0.32       250
           1       0.54      0.94      0.69       250

    accuracy                           0.57       500
   macro avg       0.66      0.57      0.50       500
weighted avg       0.66      0.57      0.50       500


=== Confusion Matrix for manipulation ===
    0    1
0  50  200
1  14  236

=== Accuracy for manipulation: 57.20% ===

=== Label Distribution ===
True labels:
true_label
1    250
0    250
Name: count, dtype: int64
Predicted labels:
pred_label
1    436
0     64
Name: count, dtype: int64

=== CoT Performance Metrics ===
- Avg tokens per call: 529.5
- Avg latency per call: 4.04s
- Total calls: 500

## Role playing with Chain-of-Thoughts

In [13]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from profiles.profile_sets import PERSON_ETHNICS

case_name_set = ["stereotype"]  # "stereotype", "manipulation"
strategy = "optimized"          # "optimized"
max_tokens = 700
role_playing_mode = "passive"
selected_profiles = ["profile4", "profile5", ]# "profile6", "profile60", "profile44", "profile45",
selected_profiles_next_priority = [
                     "profile11", "profile12", "profile13", "profile14", "profile15", 
                     ]

os.makedirs(f"results/{model_filename}/cot/reasoning", exist_ok=True)

try:
    for profile_n in selected_profiles:
        for case_name in case_name_set:
            print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) - {profile_n} ===")
    
            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
            else:
                raise ValueError(f"Unknown case name: {case_name}")
    
            cot_classifier = ChainOfThoughts(
                case=case,
                client=client,
                model=model,
                max_tokens=max_tokens,
                task_definition=task_definition,
                person_key=profile_n,
                role_playing=role_playing_mode, 
                person_set=PERSON_ETHNICS
            )
    
            rows = []
            detailed_reasoning = []
    
            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
    
                try:
                    predicted_label, metrics = cot_classifier.classify_with_strategy(
                        text, strategy=strategy
                    )
    
                    mapped_label = case.label_map.get(
                        predicted_label.strip(), list(case.label_map.values())[-1]
                    )
    
                    results = {
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped_label,
                        "raw_pred_label": predicted_label,
                        "max_tokens": cot_classifier.max_tokens,
                        "tokens_used": metrics.get("tokens_used"),
                        "prompt_tokens": metrics.get("prompt_tokens"),
                        "completion_tokens": metrics.get("completion_tokens"),
                        "latency": metrics.get("latency"),
                        "strategy": strategy,
                    }
                    rows.append(results)
    
                    raw_resp = metrics.get("raw_response", "")
                    steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                    reasoning_detail = {
                        "sample_id": idx,
                        "raw_response": raw_resp,
                        "parsed_steps": steps,
                        "analysis": analysis,       
                        "final_line": final_line,
                        "final_label": predicted_label,
                        "mapped_label": mapped_label,
                    }
                    detailed_reasoning.append(reasoning_detail)
    
                except Exception as e:
                    print(f"Error processing sample {idx}: {e}")
                    continue
    
            if not rows:
                print(f"No successful classifications for {case_name}")
                continue
    
            output_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/results_{case_name.lower()}_{strategy}_cot.csv"
            reasoning_file = f"results/{model_filename}/cot/role_playing_ethnics/{profile_n}_{role_playing_mode}/reasoning_{case_name.lower()}_{strategy}_cot.json"

            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)
    
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file} ===")
    
            with open(reasoning_file, 'w', encoding='utf-8') as f:
                json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
            print(f"=== Saved detailed reasoning to {reasoning_file} ===")
    

            try:
                if case_name.lower() == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                elif case_name.lower() == "stereotype":
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
    
                print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
                print(classification_report(y_true, y_pred, zero_division=0))
                print(f"\n=== Confusion Matrix for {case_name} ===")
                labels = sorted(set(y_true) | set(y_pred))
                print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
    
                accuracy = (y_true == y_pred).mean()
                print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")
    
                print(f"\n=== Label Distribution ===")
                print("True labels:")
                print(pd.Series(y_true).value_counts())
                print("Predicted labels:")
                print(pd.Series(y_pred).value_counts())
    
            except Exception as e:
                print(f"Error in evaluation for {case_name}: {e}")
    
            metrics = cot_classifier.get_metrics()
            print(f"\n=== CoT Performance Metrics ===")
            print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
            print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
            print(f"- Total calls: {metrics['total_calls']}")
    
    
            print(f"\n=== Sample Reasoning (raw/parsed) ===")
            for i, reasoning in enumerate(detailed_reasoning[:3]):
                print(f"\nSample {reasoning['sample_id']}:")
                print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
                print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
                if reasoning['parsed_steps']:
                    print("Parsed Steps:")
                    for step in reasoning['parsed_steps']:
                        print(f"  Step {step['step']}: {step['content'][:150]}...")
                else:
                    print("No explicit 'Step N' blocks found.")
                if reasoning['final_line']:
                    print(f"Final Line: {reasoning['final_line']}")
                print("Raw Response (truncated):")
                print((reasoning['raw_response'] or "")[:300] + "...")
                print("-" * 50)

    
except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")


=== Running Chain of Thought for stereotype (optimized) - profile4 ===


Processing stereotype: 100%|██████████| 500/500 [21:22<00:00,  2.56s/it]


=== Saved 500 rows to results/openai_4.1_mini/cot/role_playing_ethnics/profile4_passive/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/role_playing_ethnics/profile4_passive/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.72      0.80      0.76       250
   unrelated       0.77      0.69      0.73       250

    accuracy                           0.74       500
   macro avg       0.75      0.74      0.74       500
weighted avg       0.75      0.74      0.74       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         199         51
unrelated           77        173

=== Accuracy for stereotype: 74.40% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    276

Processing stereotype: 100%|██████████| 500/500 [20:20<00:00,  2.44s/it]

=== Saved 500 rows to results/openai_4.1_mini/cot/role_playing_ethnics/profile5_passive/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/role_playing_ethnics/profile5_passive/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.71      0.78      0.74       250
   unrelated       0.75      0.68      0.71       250

    accuracy                           0.73       500
   macro avg       0.73      0.73      0.73       500
weighted avg       0.73      0.73      0.73       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         194         56
unrelated           81        169

=== Accuracy for stereotype: 72.60% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    275

In [ ]:
from tqdm import tqdm
import os, json, re
import pandas as pd
from chain_of_thought import ChainOfThoughts
from stereotype_definitions import stereotype_definition_short_binary
from manipulation_definitions import manipulation_definition_short
from sklearn.metrics import classification_report, confusion_matrix
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
from profiles.profile_sets import PERSON_ETHNICS

case_name_set = ["stereotype"]  # "stereotype", "manipulation"
strategy = "optimized"          # "optimized"
max_tokens = 700
role_playing_mode = "passive"
selected_profiles = ["profile44", "profile45","profile60","profile11", "profile12", "profile13", "profile14","profile15"]
selected_profiles_next_priority = []

os.makedirs(f"results/{model_filename}/cot/reasoning", exist_ok=True)

try:
    for profile_n in selected_profiles:
        for case_name in case_name_set:
            print(f"\n=== Running Chain of Thought for {case_name} ({strategy}) - {profile_n} ===")
    
            if case_name.lower() == "manipulation":
                case = manipulation_case
                task_definition = manipulation_definition_short
                data = sample_mentalmanip
            elif case_name.lower() == "stereotype":
                case = stereotypes_case
                task_definition = stereotype_definition_short_binary
                data = sample_mgsd
            else:
                raise ValueError(f"Unknown case name: {case_name}")
    
            cot_classifier = ChainOfThoughts(
                case=case,
                client=client,
                model=model,
                max_tokens=max_tokens,
                task_definition=task_definition,
                person_key=profile_n,
                role_playing=role_playing_mode, 
                person_set=PERSON_ETHNICS
            )
    
            rows = []
            detailed_reasoning = []
    
            for idx, row in tqdm(data.iterrows(), total=len(data), desc=f"Processing {case_name}"):
                text = row[case.input_col]
                true_label = row[case.label_col]
                if isinstance(true_label, str):
                    true_label = true_label.strip()
    
                try:
                    predicted_label, metrics = cot_classifier.classify_with_strategy(
                        text, strategy=strategy
                    )
    
                    mapped_label = case.label_map.get(
                        predicted_label.strip(), list(case.label_map.values())[-1]
                    )
    
                    results = {
                        "sample_id": idx,
                        "text": text,
                        "true_label": true_label,
                        "pred_label": mapped_label,
                        "raw_pred_label": predicted_label,
                        "max_tokens": cot_classifier.max_tokens,
                        "tokens_used": metrics.get("tokens_used"),
                        "prompt_tokens": metrics.get("prompt_tokens"),
                        "completion_tokens": metrics.get("completion_tokens"),
                        "latency": metrics.get("latency"),
                        "strategy": strategy,
                    }
                    rows.append(results)
    
                    raw_resp = metrics.get("raw_response", "")
                    steps, analysis, final_line = cot_classifier._parse_steps_and_final(raw_resp)
                    reasoning_detail = {
                        "sample_id": idx,
                        "raw_response": raw_resp,
                        "parsed_steps": steps,
                        "analysis": analysis,       
                        "final_line": final_line,
                        "final_label": predicted_label,
                        "mapped_label": mapped_label,
                    }
                    detailed_reasoning.append(reasoning_detail)
    
                except Exception as e:
                    print(f"Error processing sample {idx}: {e}")
                    continue
    
            if not rows:
                print(f"No successful classifications for {case_name}")
                continue
    
            output_file = f"results/{model_filename}/cot/role_playing/{profile_n}_{role_playing_mode}/results_{case_name.lower()}_{strategy}_cot.csv"
            reasoning_file = f"results/{model_filename}/cot/role_playing/{profile_n}_{role_playing_mode}/reasoning_{case_name.lower()}_{strategy}_cot.json"

            os.makedirs(os.path.dirname(output_file), exist_ok=True)
            os.makedirs(os.path.dirname(reasoning_file), exist_ok=True)
    
            df_out = pd.DataFrame(rows)
            df_out.to_csv(output_file, index=False)
            print(f"=== Saved {len(df_out)} rows to {output_file} ===")
    
            with open(reasoning_file, 'w', encoding='utf-8') as f:
                json.dump(detailed_reasoning, f, indent=2, ensure_ascii=False)
            print(f"=== Saved detailed reasoning to {reasoning_file} ===")
    

            try:
                if case_name.lower() == "manipulation":
                    y_true = df_out["true_label"].astype(int)
                    y_pred = df_out["pred_label"].astype(int)
                elif case_name.lower() == "stereotype":
                    y_true = df_out["true_label"].astype(str).str.strip().str.lower()
                    y_pred = df_out["pred_label"].astype(str).str.strip().str.lower()
    
                print(f"\n=== Classification Report for {case_name} ({strategy}) ===")
                print(classification_report(y_true, y_pred, zero_division=0))
                print(f"\n=== Confusion Matrix for {case_name} ===")
                labels = sorted(set(y_true) | set(y_pred))
                print(pd.DataFrame(confusion_matrix(y_true, y_pred, labels=labels), index=labels, columns=labels))
    
                accuracy = (y_true == y_pred).mean()
                print(f"\n=== Accuracy for {case_name}: {accuracy:.2%} ===")
    
                print(f"\n=== Label Distribution ===")
                print("True labels:")
                print(pd.Series(y_true).value_counts())
                print("Predicted labels:")
                print(pd.Series(y_pred).value_counts())
    
            except Exception as e:
                print(f"Error in evaluation for {case_name}: {e}")
    
            metrics = cot_classifier.get_metrics()
            print(f"\n=== CoT Performance Metrics ===")
            print(f"- Avg tokens per call: {metrics['avg_tokens_per_call']:.1f}")
            print(f"- Avg latency per call: {metrics['avg_latency_per_call']:.2f}s")
            print(f"- Total calls: {metrics['total_calls']}")
    
    
            print(f"\n=== Sample Reasoning (raw/parsed) ===")
            for i, reasoning in enumerate(detailed_reasoning[:3]):
                print(f"\nSample {reasoning['sample_id']}:")
                print(f"True Label: {data.loc[reasoning['sample_id']][case.label_col]}")
                print(f"Predicted: {reasoning['final_label']} -> Mapped: {reasoning['mapped_label']}")
                if reasoning['parsed_steps']:
                    print("Parsed Steps:")
                    for step in reasoning['parsed_steps']:
                        print(f"  Step {step['step']}: {step['content'][:150]}...")
                else:
                    print("No explicit 'Step N' blocks found.")
                if reasoning['final_line']:
                    print(f"Final Line: {reasoning['final_line']}")
                print("Raw Response (truncated):")
                print((reasoning['raw_response'] or "")[:300] + "...")
                print("-" * 50)

    
except Exception as e:
    print(f"== Error: {e} ==")
    import traceback
    traceback.print_exc()

print("\n=== Chain of Thought Evaluation Complete ===")


=== Running Chain of Thought for stereotype (optimized) - profile56 ===


Processing stereotype: 100%|██████████| 500/500 [24:35<00:00,  2.95s/it]


=== Saved 500 rows to results/openai_4.1_mini/cot/role_playing/profile56_passive/results_stereotype_optimized_cot.csv ===
=== Saved detailed reasoning to results/openai_4.1_mini/cot/role_playing/profile56_passive/reasoning_stereotype_optimized_cot.json ===

=== Classification Report for stereotype (optimized) ===
              precision    recall  f1-score   support

  stereotype       0.72      0.78      0.75       250
   unrelated       0.76      0.69      0.73       250

    accuracy                           0.74       500
   macro avg       0.74      0.74      0.74       500
weighted avg       0.74      0.74      0.74       500


=== Confusion Matrix for stereotype ===
            stereotype  unrelated
stereotype         196         54
unrelated           77        173

=== Accuracy for stereotype: 73.80% ===

=== Label Distribution ===
True labels:
true_label
unrelated     250
stereotype    250
Name: count, dtype: int64
Predicted labels:
pred_label
stereotype    273
unrelated    

Processing stereotype: 100%|██████████| 500/500 [27:20<00:00,  3.28s/it]


In [16]:
# === Role-playing CoT via OpenAI Batch (Notebook version; no CLI) ===
# - Prompts are EXACTLY the ones produced by ChainOfThoughts._build_optimized_cot_prompt.
# - Each request has its own per-profile system message (passive/active).
# - Skips profiles already completed for a given case (based on existing CSV path).
# - Results written to: results/{model_filename}/cot/role_playing/{profile}_{role}/...
# - Use: submissions = submit_roleplay_cot_batches(case_name="stereotype", profiles=[...], role="passive")
#        collect_batch_outputs(submissions[0]["batch_id"])

import os, json, datetime
from typing import List, Dict, Any, Optional
from dotenv import load_dotenv
import pandas as pd
import openai

# === Project imports (must exist in your repo) ===
from chain_of_thought import ChainOfThoughts
from profiles.profile_message import make_system_message
from profiles.profile_sets import PERSON_ETHNICS
from cases.stereotypes_case import stereotypes_case
from cases.manipulation_case import manipulation_case
# If you later add MMLU:
# from cases.mmlu_case import mmlu_case

# === Environment & OpenAI client ===
load_dotenv()
if not os.getenv("API_KEY_OPENAI"):
    raise ValueError("Missing OpenAI API key: set API_KEY_OPENAI in your environment.")
client = openai.OpenAI(api_key=os.getenv("API_KEY_OPENAI"))

# === Global config (uses your existing vars if already defined in the kernel) ===
model = globals().get("model", "gpt-4.1-mini")
model_filename = globals().get("model_filename", "openai_4.1_mini")
strategy = globals().get("strategy", "optimized")
max_tokens = int(globals().get("max_tokens", 700))

# === Dataset accessors (expect your dataframes already loaded in the kernel) ===
def get_dataset(case_name: str) -> pd.DataFrame:
    if case_name == "stereotype":
        if "sample_mgsd" not in globals():
            raise ValueError("sample_mgsd is not defined in the notebook globals.")
        return globals()["sample_mgsd"]
    elif case_name == "manipulation":
        if "sample_mentalmanip" not in globals():
            raise ValueError("sample_mentalmanip is not defined in the notebook globals.")
        return globals()["sample_mentalmanip"]
    # elif case_name == "mmlu":
    #     if "sample_mmlu" not in globals():
    #         raise ValueError("sample_mmlu is not defined in the notebook globals.")
    #     return globals()["sample_mmlu"]
    else:
        raise ValueError(f"Unknown case '{case_name}'")

def get_case(case_name: str):
    if case_name == "stereotype":
        return stereotypes_case
    elif case_name == "manipulation":
        return manipulation_case
    # elif case_name == "mmlu":
    #     return mmlu_case
    else:
        raise ValueError(f"Unknown case '{case_name}'")

# === Paths & small utils ===
def _timestamp() -> str:
    return datetime.datetime.utcnow().strftime("%Y%m%dT%H%M%SZ")

def _ensure_dir(p: str):
    os.makedirs(p, exist_ok=True)

def output_paths(case_name: str, profile: str, role: str):
    base = f"results/{model_filename}/cot/role_playing_ethnics/{profile}_{role}"
    _ensure_dir(base)
    csv_path = f"{base}/results_{case_name.lower()}_{strategy}_cot.csv"
    reasoning_path = f"{base}/reasoning_{case_name.lower()}_{strategy}_cot.json"
    return csv_path, reasoning_path

def is_profile_done(case_name: str, profile: str, role: str) -> bool:
    csv_path, _ = output_paths(case_name, profile, role)
    return os.path.exists(csv_path)

def filter_profiles_not_done(case_name: str, profiles: List[str], role: str) -> List[str]:
    return [p for p in profiles if not is_profile_done(case_name, p, role)]

def build_exact_user_prompt(case, text: str) -> str:
    tmp = ChainOfThoughts(
        case=case, client=None, model=model, max_tokens=max_tokens,
        task_definition=None, person_key=None, role_playing="passive", person_set=PERSON_ETHNICS
    )
    return tmp._build_optimized_cot_prompt(text)

def build_system_message(case_name: str, profile: str, role: str) -> str:
    if role in ("active", "passive"):
        sm = make_system_message(case_name=case_name, person_key=profile, person_set=PERSON_ETHNICS)
        return sm["content"] if isinstance(sm, dict) and "content" in sm else str(sm)
    return f"You are an expert classifier for {case_name}. Think carefully and follow the prompt."

def _write_jsonl_lines(lines: List[dict], path: str):
    _ensure_dir(os.path.dirname(path))
    with open(path, "w", encoding="utf-8") as f:
        for line in lines:
            f.write(json.dumps(line, ensure_ascii=False) + "\n")
    return path

# === Public: submit batches from the notebook ===
def submit_roleplay_cot_batches(
    case_name: str,
    profiles: List[str],
    role: str = "passive",
    chunk_size: Optional[int] = None,
    skip_completed: bool = True,
) -> List[Dict[str, str]]:
    """
    Build JSONL request(s) and submit OpenAI Batch jobs.
    Returns: [{"batch_id": ..., "jsonl_path": ...}, ...]
    """
    df = get_dataset(case_name)
    case = get_case(case_name)

    if skip_completed:
        profiles = filter_profiles_not_done(case_name, profiles, role)
    if not profiles:
        print(f"Nothing to submit for '{case_name}' (all selected profiles already done).")
        return []

    print(f"Submitting {len(profiles)} profile(s) for case='{case_name}' role='{role}' using model='{model}'.")

    # Build request objects (EXACT prompts; per-profile system message)
    req_lines: List[dict] = []
    for profile in profiles:
        sys_msg = build_system_message(case.case_name, profile, role)
        for idx, row in df.iterrows():
            user_prompt = build_exact_user_prompt(case, row[case.input_col])
            custom_id = f"{case_name}:{profile}:sample_{int(idx)}"
            req_lines.append({
                "custom_id": custom_id,
                "method": "POST",
                "url": "/v1/chat/completions",
                "body": {
                    "model": model,               # exact model string
                    "temperature": 0,
                    "max_tokens": max_tokens,     # keep your CoT budget
                    "messages": [
                        {"role": "system", "content": sys_msg},
                        {"role": "user", "content": user_prompt}
                    ]
                }
            })

    # Optional chunking (very large JSONL files can be split)
    chunk_size = chunk_size or len(req_lines)
    chunks = [req_lines[i:i+chunk_size] for i in range(0, len(req_lines), chunk_size)]

    submissions = []
    for ci, chunk in enumerate(chunks, start=1):
        jsonl_path = f"batch_jobs/{case_name}_{role}_{_timestamp()}_{ci:03d}.jsonl"
        _write_jsonl_lines(chunk, jsonl_path)

        file_obj = client.files.create(file=open(jsonl_path, "rb"), purpose="batch")
        batch = client.batches.create(
            input_file_id=file_obj.id,
            endpoint="/v1/chat/completions",
            completion_window="24h",
            metadata={"case": case_name, "role": role, "chunk": str(ci)}
        )
        print(f"[SUBMITTED] batch_id={batch.id}  jsonl={jsonl_path}  requests={len(chunk)}")

        # Save a tiny manifest per batch for collection later
        manifest = {
            "batch_id": batch.id,
            "jsonl_path": jsonl_path,
            "case_name": case_name,
            "role": role,
            "model": model,
            "strategy": strategy,
            "model_filename": model_filename,
            "submitted_at": _timestamp(),
            "profiles": profiles,
        }
        _ensure_dir("batch_jobs/manifests")
        with open(f"batch_jobs/manifests/{batch.id}.manifest.json", "w", encoding="utf-8") as mf:
            json.dump(manifest, mf, indent=2, ensure_ascii=False)

        submissions.append({"batch_id": batch.id, "jsonl_path": jsonl_path})

    return submissions

# === Helpers: list batches & manifests (nice for notebooks) ===
def list_recent_batches(limit: int = 50):
    batches = client.batches.list(limit=limit)
    return [{"id": b.id, "status": b.status, "metadata": getattr(b, "metadata", {})} for b in batches]

def list_manifests() -> List[str]:
    p = "batch_jobs/manifests"
    if not os.path.isdir(p): return []
    return [os.path.join(p, f) for f in os.listdir(p) if f.endswith(".manifest.json")]

# === Public: collect a finished batch and write your usual CSV/JSON ===
def collect_batch_outputs(batch_id: str):
    manifest_path = f"batch_jobs/manifests/{batch_id}.manifest.json"
    if not os.path.exists(manifest_path):
        raise ValueError(f"Manifest not found for batch {batch_id} at {manifest_path}")

    with open(manifest_path, "r", encoding="utf-8") as f:
        manifest = json.load(f)

    case_name = manifest["case_name"]
    role = manifest["role"]
    case = get_case(case_name)
    df = get_dataset(case_name)

    batch = client.batches.retrieve(batch_id)
    if batch.status != "completed":
        print(f"Batch {batch_id} not completed yet (status={batch.status}).")
        return

    # Download results JSONL
    raw = client.files.content(batch.output_file_id).read().decode("utf-8")
    lines = [json.loads(l) for l in raw.splitlines() if l.strip()]

    # Prepare per-profile aggregation
    per_profile_rows: Dict[str, List[Dict[str, Any]]] = {}
    per_profile_reasoning: Dict[str, List[Dict[str, Any]]] = {}

    # Use your class for consistent parsing/label mapping
    cot_parser = ChainOfThoughts(
        case=case, client=None, model=manifest["model"], max_tokens=max_tokens,
        task_definition=None, person_key=None, role_playing="none", person_set=PERSON_ETHNICS
    )

    for item in lines:
        cid = item.get("custom_id")
        if not cid: 
            continue
        try:
            _case, profile, sample_part = cid.split(":")
            sample_id = int(sample_part.replace("sample_", ""))
        except Exception:
            print(f"WARNING: unexpected custom_id format: {cid}")
            continue

        resp = item.get("response", {}) or {}
        body = resp.get("body", {}) or {}
        choices = body.get("choices", []) or []
        if not choices:
            print(f"WARNING: no choices for {cid}")
            continue

        text = (choices[0]["message"]["content"] or "").strip()
        usage = body.get("usage", {}) or {}

        # Label norm + steps parsing
        final_label = cot_parser._extract_label(text, case.valid_labels)
        mapped_label = case.label_map.get(final_label.strip(), list(case.label_map.values())[-1])
        steps, analysis, final_line = cot_parser._parse_steps_and_final(text)

        true_label = df.loc[sample_id, case.label_col]
        if isinstance(true_label, str):
            true_label = true_label.strip()
        sample_text = df.loc[sample_id, case.input_col]

        per_profile_rows.setdefault(profile, []).append({
            "sample_id": sample_id,
            "text": sample_text,
            "true_label": true_label,
            "pred_label": mapped_label,
            "raw_pred_label": final_label,
            "max_tokens": max_tokens,
            "tokens_used": usage.get("total_tokens"),
            "prompt_tokens": usage.get("prompt_tokens"),
            "completion_tokens": usage.get("completion_tokens"),
            "latency": None,
            "strategy": strategy,
        })
        per_profile_reasoning.setdefault(profile, []).append({
            "sample_id": sample_id,
            "raw_response": text,
            "parsed_steps": steps,
            "analysis": analysis,
            "final_line": final_line,
            "final_label": final_label,
            "mapped_label": mapped_label,
        })

    # Write per-profile outputs in your standard layout
    for profile, rows in per_profile_rows.items():
        rows = sorted(rows, key=lambda r: r["sample_id"])
        csv_path, reasoning_path = output_paths(case_name, profile, role)
        pd.DataFrame(rows).to_csv(csv_path, index=False)
        with open(reasoning_path, "w", encoding="utf-8") as f:
            json.dump(per_profile_reasoning[profile], f, indent=2, ensure_ascii=False)
        print(f"[WROTE] {len(rows)} rows → {csv_path}")
        print(f"[WROTE] reasoning → {reasoning_path}")

    print(f"[DONE] Collected batch {batch_id}.")


In [17]:
subs = submit_roleplay_cot_batches(
    case_name="stereotype",
    profiles=[f"profile{i}" for i in range(1, 61)],
    role="passive",
    chunk_size=3000,   # or e.g. 2000 to split into multiple batch files
)

Submitting 43 profile(s) for case='stereotype' role='passive' using model='gpt-4.1-mini'.
[SUBMITTED] batch_id=batch_68af79c14f5081909cfbefe929fff55c  jsonl=batch_jobs/stereotype_passive_20250827T213324Z_001.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af79ddac7881909e6648f888751082  jsonl=batch_jobs/stereotype_passive_20250827T213353Z_002.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af79f99f588190a0a35cd52e7e2fde  jsonl=batch_jobs/stereotype_passive_20250827T213422Z_003.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7a1342e481908e7818bfb1c98096  jsonl=batch_jobs/stereotype_passive_20250827T213449Z_004.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7a30ce688190b9afc8c56547a430  jsonl=batch_jobs/stereotype_passive_20250827T213515Z_005.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7a4e382c8190b0c34a7c9970f531  jsonl=batch_jobs/stereotype_passive_20250827T213545Z_006.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7a69572c8190b666ec2ccdd60b93  jsonl=batch_job

In [18]:
subs = submit_roleplay_cot_batches(
    case_name="manipulation",
    profiles=[f"profile{i}" for i in range(1, 61)],
    role="passive",
    chunk_size=3000,   # or e.g. 2000 to split into multiple batch files
)

Submitting 60 profile(s) for case='manipulation' role='passive' using model='gpt-4.1-mini'.
[SUBMITTED] batch_id=batch_68af7a98bb68819083ea09619bc1cf8b  jsonl=batch_jobs/manipulation_passive_20250827T213648Z_001.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7aba7a2c8190aabd8909496f95fe  jsonl=batch_jobs/manipulation_passive_20250827T213728Z_002.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7ade9ba08190821788fdfa1ca1e4  jsonl=batch_jobs/manipulation_passive_20250827T213802Z_003.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7afb5d948190a462ba8b5d1141c3  jsonl=batch_jobs/manipulation_passive_20250827T213839Z_004.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7b1e41f48190bb22ca5105f95c10  jsonl=batch_jobs/manipulation_passive_20250827T213907Z_005.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7b4253788190be8458e1a3b25295  jsonl=batch_jobs/manipulation_passive_20250827T213942Z_006.jsonl  requests=3000
[SUBMITTED] batch_id=batch_68af7b6214e081908dcc16c28a4b33bd  j

KeyboardInterrupt: 